## Plots for the figures of the paper (revolving only simulations)

In [ ]:
from pathlib import Path
import importlib

import file_funcs
import plot_funcs

## Success matrices - pos force and combined diagrams

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches

import colors

importlib.reload(file_funcs)

H = 4
N = 2**H

folder_pos = Path("Training\\May17Like_noFlipChain")
folder_force = Path("Training\\Apr23randomPosAfterFroceExplode")

M_pos, _, _ = file_funcs.build_success_matrix(
    folder_pos, N=N, old=False, near_miss=False, symmetry=False,
    omit_inverted=True, find_symmetrical=False,
)
M_F, _, _ = file_funcs.build_success_matrix(
    folder_force, N=N, old=False, near_miss=False, symmetry=False,
    omit_inverted=True, find_symmetrical=False,
)

M_pos_includeSymm = copy.copy(M_pos)
M_pos_includeSymm[M_pos[::-1, ::-1] == 0] = 0

M_F_includeSymm = copy.copy(M_F)
M_F_includeSymm[M_F[::-1, ::-1] == 0] = 0

# Both position and force
M_both = copy.copy(M_pos_includeSymm)
M_both[M_F_includeSymm == 0] = 0
M_both[M_both[::-1, ::-1] == 0] = 0


In [ ]:

labels = [format(i, f"0{H}b") for i in range(N)]
_, _, custom_cmap = colors.color_scheme()
font_size = 18

fig, axes = plt.subplots(
    1, 3, figsize=(13.5, 4.8), sharey=True, constrained_layout=True,
)
for ax, matrix, title in zip(
    axes,
    (M_pos_includeSymm, M_F_includeSymm, M_both),
    ("Position", "Force", "Combined"),
):
    matrix_masked = np.ma.masked_where(np.eye(N, dtype=bool), matrix)
    image = ax.imshow(
        matrix_masked, cmap=custom_cmap, vmin=0, vmax=4,
        origin="lower", interpolation="none", aspect="equal",
    )
    ax.set_xticks(range(N), labels, rotation=90)
    ax.set_yticks(range(N), labels)
    ax.set_xlabel("desired buckle", fontsize=font_size)
    ax.set_title(title, fontsize=font_size, pad=9)
    ax.tick_params(axis="both", labelsize=font_size)

axes[0].set_ylabel("initial buckle", fontsize=font_size)
for ax in axes[1:]:
    ax.tick_params(axis="y", left=False, labelleft=False)

legend_handles = [
    patches.Patch(facecolor=custom_cmap(image.norm(0)), label="Success"),
    # patches.Patch(facecolor=custom_cmap(image.norm(1)), label="Missing"),
    patches.Patch(facecolor=custom_cmap(image.norm(2)), label="Failure"),
]
axes[-1].legend(
    handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1),
    fontsize=font_size, frameon=False, borderaxespad=0,
)
plt.show()

success_rate_both = np.sum(M_both == 0) / (np.sum(M_both == 2) + np.sum(M_both == 0))
print("success_rate_both=", success_rate_both)


## Efficiently covering transitions

In [ ]:
# Hamming-transition coverage from exported training or tip-sweep files
from pathlib import Path
import pandas as pd

importlib.reload(file_funcs)

folder_pos = Path("efficient_transitions\\May17Like_noFlipChain")
folder_F = Path("efficient_transitions\\Apr23randomPosAfterFroceExplode")
folder_rand = Path("efficient_transitions\\July16_grid_sweep")

coverage_pos = pd.read_csv(folder_pos / "cumulative_transition.csv")
coverage_F = pd.read_csv(folder_F / "cumulative_transition.csv")
coverage_rand = pd.read_csv(folder_rand / "cumulative_transition.csv")


In [ ]:
importlib.reload(plot_funcs)

import matplotlib.pyplot as plt

fig, (ax_left, ax_right) = plt.subplots(
    1, 2, figsize=(8, 4), sharey=True,
    gridspec_kw={"width_ratios": (3, 1), "wspace": 0.06},
)
for coverage_df, x_col, label, color_index in (
    (coverage_pos, "training_task", "Position", 0),
    (coverage_F, "training_task", "Force", 1),
    (coverage_rand, "step", "Random", 2),
):
    for ax in (ax_left, ax_right):
        plot_funcs.plot_cumulative_transition_curve(
            coverage_df,
            x_col=x_col,
            x_label="",
            label=label if ax is ax_left else None,
            color_index=color_index,
            ax=ax,
        )

ax_left.set_xlim(0, 240)
ax_right.set_xlim(2260, 2310)  # The random sweep currently ends at step 2304.
ax_left.set_xticks([0, 40, 80, 120, 160, 200])
ax_right.set_xticks([2260, 2290, 2310])
ax_left.spines["right"].set_visible(False)
ax_right.spines["left"].set_visible(False)
ax_right.set_ylabel("")
ax_right.tick_params(axis="y", left=False, labelleft=False)

# Diagonal marks indicate the omitted x range.
break_size = 0.018
break_style = dict(color="k", clip_on=False, linewidth=1.2)
ax_left.plot((1 - break_size, 1 + break_size), (-break_size, +break_size),
             transform=ax_left.transAxes, **break_style)
ax_right.plot((-break_size, +break_size), (-break_size, +break_size),
              transform=ax_right.transAxes, **break_style)
fig.supxlabel("training task / sweep step", y=0.01)

fig.subplots_adjust(bottom=0.18)
fig.savefig(Path("Figures") / "cumulative_transition.png", dpi=200, bbox_inches="tight")
plt.show()

## Transitions on ring - init to fin

In [ ]:
importlib.reload(file_funcs)

folder_force = Path("Training\\Apr23randomPosAfterFroceExplode")
# folder = Path("Training\\May17Pos_symmetrical_delta")
folder_pos = Path("Training\\May17Like_noFlipChain")
folder_sweep = Path("grid sweep\\separate files\\tip_grid_transition_csvs")
only_reached_nodes = False
only_init_and_final_buckles = True
omit_inverted = False
transition_mode = "ring"  # "hamming" for one-bit missing edges; "ring" for all-to-all missing edges
reciprocity = False  # Add each transition count to the bitwise sign-opposite transition
transitions_pos, _, _, edge_zero_loss_count_pos, missing_edges_pos = file_funcs.buckle_transitions(folder_pos,
                                                                                                   only_init_and_final_buckles=only_init_and_final_buckles,
                                                                                                   omit_inverted = omit_inverted,
                                                                                                   transition_mode=transition_mode,
                                                                                                   reciprocity=reciprocity)
transitions_F, _, _, edge_zero_loss_count_F, missing_edges_F = file_funcs.buckle_transitions(folder_force,
                                                                                             only_init_and_final_buckles=only_init_and_final_buckles,
                                                                                             omit_inverted = omit_inverted,
                                                                                             transition_mode=transition_mode,
                                                                                             reciprocity=reciprocity)
transitions_swp, _, _, edge_zero_loss_count_swp, missing_edges_swp = file_funcs.buckle_transitions(folder_sweep,
                                                                                                   only_init_and_final_buckles=only_init_and_final_buckles,
                                                                                                   omit_inverted = omit_inverted,
                                                                                                   transition_mode=transition_mode,
                                                                                                   reciprocity=reciprocity)


In [ ]:
importlib.reload(plot_funcs)
export_png = True
export_eps = False
export_pdf = True

fig, axes = plot_funcs.plot_ring_transition_diagrams(
    transitions_pos,
    transitions_F,
    transitions_swp,
    edge_zero_loss_counts=(edge_zero_loss_count_pos, edge_zero_loss_count_F, edge_zero_loss_count_swp),
    missing_edges=(missing_edges_pos, missing_edges_F, missing_edges_swp),
    titles=("Position", "Force", "Random"),
    transitions_between_runs=only_init_and_final_buckles,
    only_reached_nodes=only_reached_nodes,
    save_stem=Path("Figures") / "ring_transition_diagrams",
    export_png=export_png,
    export_eps=export_eps,
    export_pdf=export_pdf,
    dpi=600,
)

## Accuracy a.f.o. H + loss and Hamming improvement

In [ ]:
import numpy as np

H = np.array([4, 5, 6])
accuracy = np.array([0.9, 0.64, 0.34])

folder_H5 = Path("Training\\July10_and_12_H5_safetyMargin_pos_and_recip")
# folder_H5_reciprocal = Path("Training\\July12_reciprocalTo_July10_H5_safetyMargin")
folder_H6 = Path("Training\\July19_H6_pos_pt25LEMargin")

# loss_H5_a, Hamming_H5_a, buckle_pairs_H5_a = file_funcs.build_loss_columns(folder_H5, include_symm=True)
# loss_H5_b, Hamming_H5_b, buckle_pairs_H5_b = file_funcs.build_loss_columns(folder_H5_reciprocal)
# loss_H5 = np.vstack((loss_H5_a, loss_H5_b))
# Hamming_H5 = np.vstack((Hamming_H5_a, Hamming_H5_b))
# buckle_pairs_H5 = np.vstack((buckle_pairs_H5_a, buckle_pairs_H5_b))
loss_H5_nosymm, Hamming_H5_nosymm, _ = file_funcs.build_loss_columns(folder_H5, include_symm=False)
loss_H6_nosymm, Hamming_H6_nosymm, _ = file_funcs.build_loss_columns(folder_H6, include_symm=False)
loss_H5, Hamming_H5, buckle_pairs_H5 = file_funcs.build_loss_columns(folder_H5, include_symm=True)
loss_H6, Hamming_H6, buckle_pairs_H6 = file_funcs.build_loss_columns(folder_H6, include_symm=True)


print('loss initial t final H=5', [np.mean(loss_H5_nosymm, axis=0)[0], np.mean(loss_H5, axis=0)[1]])
print('loss initial t final H=6', [np.mean(loss_H6_nosymm, axis=0)[0], np.mean(loss_H6, axis=0)[1]])

print('Hamming initial t final H=5', [np.mean(Hamming_H5_nosymm, axis=0)[0], np.mean(Hamming_H5, axis=0)[1]])
print('Hamming initial t final H=6', [np.mean(Hamming_H6_nosymm, axis=0)[0], np.mean(Hamming_H6, axis=0)[1]])



In [ ]:
importlib.reload(plot_funcs)

fig, axes = plot_funcs.plot_accuracy_loss_hamming_summary(
    H,
    accuracy,
    loss_columns=(loss_H5, loss_H6),
    hamming_columns=(Hamming_H5, Hamming_H6),
    metric_Hs=(5, 6),
    save_path=Path("Figures") / "accuracy_loss_hamming_summary.png",
    dpi=600,
)